# Convert (Already-combined and split) Model Lat/Lon Files to Polar Stereographic Files

This notebook provides a workflow for converting lat/lon netcdf files that already include underlying data to polar stereographic coordinates. To get the data into the proper format, we needed to do the following:
 - utilize a template file as a base for modifying netcdf files of the existing mask arrays. The template file includes all proper GIS attributes.
 - python script reads the Geotiff file and creates a netcdf file with the x,y,lat,lon coordinate dimensions and variables

Authors:
--------
 - Teagan King

Prerequisites:
--------------
* Create geoTiff file (done in PolarLatLon.ipynb)
* Create SCRIP File (done in PolarLatLon.ipynb)
* Generate template files for NH & SH (done in PolarLatLon.ipynb)
* regrid files from lat/lon to polar (see instructions below)
* rename regridded files (see instructions below)

## regrid TMQ/IVT/PSL/PR data from lat/lon to polar

In [ ]:
# SH SCRIPT

# submit ncremap scripts below in batch scripts, see example at
#     /glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/2dlatlon/sh_scripts/remap_script and batch_remap.sh

# the above batch script uses the mapfile to remap srcinitfile to dstinitfile as shown below:
# ncremap -m ./map_${srcgrid}_to_${dstgrid}_near.nc -i ${srcinitfile} -o ${dstinitfile}

```find ../latlon/BRCP85C5CN -mindepth 2 -maxdepth 2 -type f > BRCP85C5CN.txt```

remap_np_script:

```
while IFS= read -r filepath; do
    relpath="${filepath#../latlon/BRCP85C5CN/}"
    outpath="nh/$relpath"
    mkdir -p "$(dirname "$outpath")"
    if [ ! -f "$outpath" ]; then
        ncremap -m map_fv0.23x0.31_to_np_stereo_near_4.nc -i "$filepath" -o "$outpath"
    else
        echo "Skipping $outpath (already exists)"
    fi
done < BRCP85C5CN.txt
```

batch_np_remap.sh:

```
#!/bin/bash -l
#PBS -N remap_np
#PBS -A P93300313
#PBS -l select=1:ncpus=1:mem=100GB
#PBS -l walltime=6:00:00
#PBS -q casper
#PBS -m abe
#PBS -M tking@ucar.edu

# This script can be used to submit batch ncremap scripts, such as those in the example psl_remap_script.
# Check the status of this script after it is submitted with `qstat -u tking`.
# Please use your own account after `-A`, and update with your email to get notifications.
# It is also recommended to update the name of the script and jobname when submitting for other variables.

module load nco/4.7.9

sh remap_np_script
```

Then run with:

`qsub batch_np_remap.sh`

## Rename and format regridded files

```
mkdir -p dropped_some  # make sure output dir exists
mkdir -p dropped_all
mkdir -p xy
mkdir -p tmp
mkdir -p renamedlon
mkdir -p renamedlat
mkdir -p almost_final
mkdir -p renamed_vars
mkdir -p final

for filename in *; do
    ncks -x -v w_stag,slat,nbnd "$filename" "dropped_some/$filename"
    ncks -C -x -v time_bnds,nvertices,lat_bnds,lon_bnds,bnds "dropped_some/$filename" "dropped_all/$filename"
    ncrename -d lon,x -d lat,y "dropped_all/$filename" "xy/$filename"
    #ncap2 -s 'longitude=lon' xy/$filename tmp/$filename
    #ncks -C -x -v lon tmp/$filename renamedlon/$filename
    #ncrename -v lat,latitude renamedlon/$filename renamedlat/$filename
    ncrename -d x,lon -d y,lat xy/$filename almost_final/$filename
    #ncrename -v TMQ,tmq almost_final/$filename final/$filename
    ncrename -v PSL,psl almost_final/$filename final/$filename
    #ncrename -v U850,u850 final/$filename
    #ncrename -v V850,v850 final/$filename
done

rm -r dropped_all
rm -r dropped_some
rm -r renamedlat
rm -r renamedlon
rm -r tmp
rm -r xy

```

flip horizontally with: `python /glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/2dlatlon/nh_polar/renamed/ivt/flip_image.py <INPUT FILE> <OUTPUT_FILE>` -- NEEDED For NH

10) any cropping necessary: /glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/2dlatlon/nh_polar/renamed/ivt/zoom.py (use 450) -- SHOULDN'T BE NECESSARY HERE BECAUSE NOT ALIGNING WITH UNDERLYING DATA? BUT COULD BE SLIGHTLY DIFFERENT REGION... USING FOR NH!